# 《PythAPCS123》單元 13-4：例外捕捉語法：try ... except 架構與未知長度輸入處理

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-4_try_except_and_eof_handling.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：掌握 Python 原生最強大的執行期防禦體系 `try ... except`，理解主動嘗試與例外攔截的控制流程跳轉機制。學會精準捕獲特定例外類別，堅決杜絕濫用裸露 `except:` 隱匿程式邏輯臭蟲的危險惡習。專攻 APCS 考場最經典、最具殺傷力的「未知行數輸入至檔案結尾（EOF）」通關密技，搞懂 `else` 與 `finally` 完整語意，並建立「何時該用 if、何時該用 try-except」的競賽架構決策力。


### 13.4.1 防禦性程式設計哲學：優雅攔截非預期狀況，避免程式強制崩潰

在上一單元中，我們認識了「Look Before You Leap（LBYL，三思而後行）」的哲學，也就是在可能發生錯誤的每一處細節，都手動寫下長長一段 `if` 條件式進行檢查。然而，在許多真實世界的計算情境以及線上評判系統中，有些外部狀況是我們「無法事先透過簡單 `if` 完全精準預測」的。例如：使用者輸入了一長串未知字串，我們很難光靠肉眼或簡單規則保證它必定能轉換成整數；或者更典型的，當測資不斷從管線湧入時，我們根本無從預知官方測資何時會「突然結束送出」。

這時，Python 引導我們走向另一種極其成熟且優雅的設計哲學——**「Easier to Ask for Forgiveness than Permission（EAFP，先斬後奏）」**。
其核心思想是：與其在每次行動前都花費昂貴的心力去做繁瑣的先驗檢查，不如大膽地「先嘗試執行（try）」正常業務邏輯；若是在執行過程中真的發生了突發例外，我們再於第一時間由安全防護網「攔截捕捉（except）」並優雅地妥善處理，防止整個應用程式崩潰陣亡。

這種「優雅降級（Graceful Degradation）」的防衛思維，是高品質工業級程式碼與 APCS 考場抗錯必備的深層素養。


In [ ]:
# 13.4.1 程式碼演示：對比裸奔崩潰 vs 優雅降級攔截
def fragile_conversion(user_input):
    print(f"嘗試轉換輸入值: {repr(user_input)}")
    # 脆弱寫法：無任何防護，一旦型態錯誤直接暴斃中斷整個腳本
    val = int(user_input)
    return val * 2

def robust_conversion(user_input):
    print(f"嘗試安全轉換輸入值: {repr(user_input)}")
    # 優雅防禦：使用 try-except 建立彈性緩衝防護網
    try:
        val = int(user_input)
        return val * 2
    except ValueError:
        print("  ⚠️ 警告：偵測到非法字元，無法轉為整數！已優雅降級回傳預設值 0")
        return 0

print("--- 1. 測試合法資料 ---")
print("結果:", robust_conversion("42"))

print("\n--- 2. 測試異常資料（優雅防禦保證程式不崩潰）---")
print("結果:", robust_conversion("Python3"))


### 13.4.1 語法重點回顧與核心觀念提煉

EAFP 與 LBYL 兩種思維模式在 Python 中各有其用武之地：
1. **先斬後奏（EAFP）優勢**：
   - 程式碼的主幹邏輯一目了然，不需要被大量的巢狀 `if-else` 防守層層包覆。
   - 避免了先驗檢查與實際執行之間的「重複運算開銷」（例如不用先檢查一次合法性，然後再做一次剖析）。
2. **優雅降級核心價值**：
   - 當例外發生時，程式不會無預警崩潰，而是可以「記錄錯誤記錄檔」、「給予預設值」或「優雅跳出迴圈」。
   - 在 APCS 考場中，這種機制是唯一能夠乾淨處理「檔案讀取結尾」的終極法寶。


In [ ]:
# 13.4.1 學生實作練習：彈性整數求和器
# 任務說明：實作 safe_sum_mixed_list(data) 函式
# 給定一個包含各類字串或數字的混合串列 data（如 ["10", "abc", "25", "3.14", "5"]）
# 嘗試將每個元素轉為整數 int() 並累加總和；若某個元素無法被 int() 解析，請優雅略過它而不中斷程式
# 回傳所有成功轉換的整數總和！

def safe_sum_mixed_list(data: list) -> int:
    total = 0
    # 請在此處使用 for 迴圈搭配 try-except 進行安全累加
    for item in data:
        try:
            total += int(item)
        except (ValueError, TypeError):
            pass
    return total

# 測試用例
mixed = ["10", "hello", "20", "world", "30"]
print("混合串列總和:", safe_sum_mixed_list(mixed))


In [ ]:
# 13.4.1 單元測試驗證
assert safe_sum_mixed_list(["1", "2", "3"]) == 6
assert safe_sum_mixed_list(["10", "abc", "20"]) == 30
assert safe_sum_mixed_list(["x", "y", "z"]) == 0
assert safe_sum_mixed_list([]) == 0
assert safe_sum_mixed_list([5, "15", None, "25"]) == 45
print("13.4.1 單元測試全數通過！")


### 13.4.2 try ... except 核心語法與流程跳轉圖解

`try ... except` 是 Python 語言內建的結構化例外處理機制。它的基本文法結構如下：

```python
try:
    # 可能會拋出例外的危險程式碼區塊（嘗試執行）
    危險操作 1
    危險操作 2
except 例外名稱:
    # 只有當 try 區塊內部拋出該特定例外時，才會跳轉執行的救援區塊
    補救措施
```

深入理解其**「控制流程跳轉（Control Flow Jump）」**機制至關重要：
1. **理想順暢路徑（Happy Path）**：直譯器進入 `try` 區塊，一行一行往下執行。若所有指令皆平穩完成、未引爆任何例外，直譯器會「完全跳過」`except` 區塊，直接接續執行後續的程式碼。
2. **觸發跳轉路徑（Exception Path）**：若在 `try` 區塊中的「某一行指令」突然引爆了相符的例外，直譯器會「立刻中斷」`try` 區塊的後續指令（該行之後的所有代碼全數被跳過！），控制權瞬間垂直跳躍至 `except` 區塊中執行救援。
3. **未被攔截路徑（Unhandled Path）**：若拋出的例外種類與 `except` 所宣告的種類不一致，防護網無效，直譯器依舊會向上拋出例外導致程式當場崩潰。


In [ ]:
# 13.4.2 程式碼演示：透視 try-except 的垂直跳躍流程
def demo_flow_jump(divisor):
    print(f"\n--- 開始執行 demo_flow_jump(divisor={divisor}) ---")
    print("步驟 [1]: 進入 try 區塊前置作業")
    try:
        print("步驟 [2]: 即將進行除法運算...")
        result = 100 / divisor
        print(f"步驟 [3]: 計算順利完成！結果為 {result}") # 若除以 0，這行會被徹底跳過！
    except ZeroDivisionError:
        print("步驟 [4]: ⚠️ 觸發跳轉！攔截到除以零例外，進入 except 救援區塊")
        result = 0
        
    print(f"步驟 [5]: 離開 try-except 結構，最終交付結果: {result}")
    return result

# 案例 A: 正常無出錯路徑 (1 -> 2 -> 3 -> 5)
demo_flow_jump(5)

# 案例 B: 出錯跳轉路徑 (1 -> 2 -> 4 -> 5，注意步驟 3 被跳過！)
demo_flow_jump(0)


### 13.4.2 語法重點回顧與核心觀念提煉

追蹤 `try ... except` 執行流程的兩大心智模型：
1. **即刻熔斷機制**：`try` 內部一旦出錯，就像電路跳電一樣立刻中斷，錯誤行下方的任何指令（例如釋放資源或重設變數）都不會被執行。因此，切勿把「無論如何都必須執行的清理動作」放在 `try` 區塊的最末端。
2. **作用域（Scope）透明**：在 `try` 或 `except` 區塊內所賦值的變數，其作用域屬於當前函式或全域，離開區塊後依然可以繼續被存取（但在 `try` 內部宣告的變數若在賦值前就出錯，該變數可能尚未定義，故通常建議在 `try` 外部先設定初始預設值）。


In [ ]:
# 13.4.2 學生實作練習：安全字串轉浮點數剖析器
# 任務說明：實作 parse_float_safely(text, default_val) 函式
# 嘗試將 text 字串轉為 float 浮點數
# 若順利轉換，回傳該浮點數
# 若因格式不符拋出 ValueError，攔截並回傳 default_val 預設值！

def parse_float_safely(text: str, default_val: float) -> float:
    # 請在此處實作 try-except 結構
    try:
        return float(text)
    except ValueError:
        return default_val

# 測試用例
print("正常解析:", parse_float_safely("3.14159", 0.0))
print("異常文字解析:", parse_float_safely("not_a_number", -1.0))


In [ ]:
# 13.4.2 單元測試驗證
assert parse_float_safely("12.5", 0.0) == 12.5
assert parse_float_safely("-8.0", 0.0) == -8.0
assert parse_float_safely("abc", 99.9) == 99.9
assert parse_float_safely("", 0.0) == 0.0
print("13.4.2 單元測試全數通過！")


### 13.4.3 精準捕捉特定例外：嚴禁濫用裸露 except: 遮蔽邏輯臭蟲

在學習 `try ... except` 時，許多初學者會因為貪圖方便，寫出所謂的**「裸露 except（Bare Except）」**：
```python
# 致命壞習慣：千萬不要這樣寫！
try:
    do_something()
except:
    pass
```

這是在軟體工程與 APCS 考場中最具破壞力的危險毒瘤，被資深工程師稱為「臭蟲消音器（Bug Silencer）」。
當你只寫 `except:` 而不指明例外名稱時，Python 會攔截「整座直譯器中的任何例外」——包括：
1. **變數拼錯造成的 NameError**：例如你把變數 `total` 不小心打成 `totall`，裸露 except 會直接把它吞掉，讓你誤以為程式執行很順利，結果算出來的答案完全是錯的（導致莫名其妙的 WA）！
2. **鍵盤強制中斷（KeyboardInterrupt）**：當你的程式陷入無窮迴圈你想按下 Ctrl+C 停機時，裸露 except 會連中斷訊號都攔截下來，導致程式完全失控無法終止。

正確的專業做法永遠是**「精準指名捕捉（Catch Specific Exceptions）」**。如果你預期可能會發生整數轉換錯誤或陣列越界，請明確使用元組打包寫下：`except (ValueError, IndexError):`；若需要印出直譯器的具體報錯文字，可使用 `as e` 取得例外物件：`except ValueError as e:`。


In [ ]:
# 13.4.3 程式碼演示：裸露 except 吞掉拼寫錯誤 vs 精準攔截排查
print("--- 壞示範：裸露 except 造成靈異現象（不知為何出錯）---")
def dangerous_calculator(x_str):
    total = 0
    try:
        val = int(x_str)
        # 故意打錯變數名：將 total 拼寫成 totall
        totall = totall + val 
    except:
        # 裸露 except 把 NameError: name 'totall' is not defined 硬生生吞掉！
        pass
    return total

print("呼叫危險計算器 ('10'):", dangerous_calculator("10"))
print("注意：輸入 10 計算結果卻依然是 0，因為內部的 NameError 臭蟲被掩蓋了！")

print("\n--- 好示範：精準指定捕捉 ValueError，讓 NameError 正常暴露 ---")
def safe_calculator(x_str):
    total = 0
    try:
        val = int(x_str)
        total += val
    except ValueError as e:
        print(f"[精準攔截] 格式錯誤被安全捕獲: {e}")
    return total

print("呼叫安全計算器 ('10'):", safe_calculator("10"))
print("呼叫安全計算器 ('abc'):", safe_calculator("abc"))


### 13.4.3 語法重點回顧與核心觀念提煉

精準例外捕捉的三大黃金紀律：
1. **絕不寫不帶名稱的裸露 `except:`**：永遠明確寫出你要防範的例外型別（例如 `except ValueError:`）。
2. **多重例外聯合防禦**：若一段程式碼可能同時引爆多種預期內的例外，可以用小括號打包：
   ```python
   except (ValueError, IndexError, KeyError) as e:
       print(f"安全防護觸發: {type(e).__name__}")
   ```
3. **階層化獨立處理**：若不同例外需要執行不同的補救措施，可以串聯多個 `except` 區塊：
   ```python
   except IndexError:
       # 處理索引越界
   except ValueError:
       # 處理格式錯誤
   ```


In [ ]:
# 13.4.3 學生實作練習：多重例外安全存取器
# 任務說明：實作 safe_lookup_and_convert(mapping, key) 函式
# 給定一個字典 mapping 與一個查詢鍵 key
# 需求：
# 1. 從字典中取出 mapping[key]（可能引爆 KeyError）
# 2. 將取出的字串值使用 int() 轉為整數（可能引爆 ValueError）
# 3. 若成功，回傳轉換後的整數
# 4. 若引發 KeyError，回傳字串 "KEY_NOT_FOUND"
# 5. 若引發 ValueError，回傳字串 "INVALID_NUMBER"
# 嚴格要求：不可使用裸露 except:，必須分開精準捕捉！

def safe_lookup_and_convert(mapping: dict, key: str):
    # 請在此處實作精準分流捕捉
    try:
        raw_val = mapping[key]
        return int(raw_val)
    except KeyError:
        return "KEY_NOT_FOUND"
    except ValueError:
        return "INVALID_NUMBER"

# 測試用例
sample_dict = {"age": "18", "score": "ninety", "level": "5"}
print("合法查詢:", safe_lookup_and_convert(sample_dict, "age"))
print("格式錯誤:", safe_lookup_and_convert(sample_dict, "score"))
print("鍵值缺失:", safe_lookup_and_convert(sample_dict, "height"))


In [ ]:
# 13.4.3 單元測試驗證
test_map = {"a": "100", "b": "invalid", "c": "0"}
assert safe_lookup_and_convert(test_map, "a") == 100
assert safe_lookup_and_convert(test_map, "c") == 0
assert safe_lookup_and_convert(test_map, "b") == "INVALID_NUMBER"
assert safe_lookup_and_convert(test_map, "z") == "KEY_NOT_FOUND"
print("13.4.3 單元測試全數通過！")


### 13.4.4 APCS 必備通關密技：try ... except EOFError: 完美讀取未知行數測資

在 APCS 與各類程式競賽（如 ZeroJudge、Codeforces）中，題目的輸入格式描述常常會出現這麼一句令新手摸不著頭緒的話：**「輸入包含多筆測試資料，每筆一行，直到檔案結束（EOF, End of File）為止。」**

當初學者嘗試使用一般的迴圈時，往往會面臨巨大困惑：「題目根本沒告訴我總共有幾行，那我到底該寫 `for _ in range(???)` 幾次？如果寫 `while True:`，迴圈不就會永遠停不下來而導致 Time Limit Exceeded (TLE) 嗎？」

這正是競技程式中最經典的難題。在自動評判系統（Online Judge）中，測資是透過「標準輸入管線（stdin）」重定向送入的。當所有測試資料全部送完時，底層的輸入串流就會抵達「檔案結尾（EOF）」。此時如果 Python 再次呼叫 `input()`，因為已經沒有任何字元可以讀取，直譯器就會當場拋出 **`EOFError: EOF when reading a line`** 例外！

這意味著：**`EOFError` 不是壞事，它正是直譯器向我們吹哨通知「測資已全數讀完，請優雅收工！」的通關密碼**。
標準的 APCS 萬用讀檔模板為：
```python
while True:
    try:
        line = input()
        # 處理每一行測資...
    except EOFError:
        break # 遇到檔案結尾，優雅跳出無窮迴圈！
```


In [ ]:
# 13.4.4 程式碼演示：模擬 APCS 未知行數讀取至 EOF 之通關模板
import io
import sys

# 模擬 Online Judge 裁判系統送入的未知行數測資串流（共 4 行資料，隨後抵達檔案結尾）
mock_oj_input = '''15 25
30 40
100 200
5 7
'''

# 將標準輸入臨時替換為模擬串流
saved_stdin = sys.stdin
sys.stdin = io.StringIO(mock_oj_input)

print("=== 模擬 APCS 評判系統開始餵入測資 ===")
case_count = 0
total_sum = 0

# APCS 考場標準未知行數通關模板：
while True:
    try:
        line = input()
        if not line.strip(): # 忽略偶爾出現的空白行
            continue
        # 業務邏輯：每行讀取兩個整數並計算總和
        a, b = map(int, line.split())
        case_count += 1
        ans = a + b
        total_sum += ans
        print(f"第 {case_count} 筆測資: {a} + {b} = {ans}")
    except EOFError:
        # 捕捉到 EOFError，代表官方測資全部讀取完畢，優雅跳出！
        print(f"[通關信號] 偵測到 EOF (檔案結尾)，順利讀完 {case_count} 筆測資！")
        break

# 還原標準輸入
sys.stdin = saved_stdin
print(f"全數計算總計: {total_sum}")


### 13.4.4 語法重點回顧與核心觀念提煉

掌握 `EOFError` 通關模板的兩大關鍵守則：
1. **`while True:` 與 `break` 完美搭檔**：利用 `while True:` 無休止地嘗試讀取下一行；一旦拋出 `EOFError`，立刻在 `except` 區塊內執行 `break`，整個讀取迴圈便會平靜且迅速地終止。
2. **多行未知輸入的另外兩種寫法（延伸認知）**：
   - 寫法 A（考場最直觀推薦）：`while True: try: line = input() except EOFError: break`
   - 寫法 B（進階極速寫法）：`import sys; for line in sys.stdin: ...`（內建支援走訪直到 EOF，效能極高）。
掌握此模板，即可在 APCS 考場面對「未知筆數」、「多測資直到 EOF」的題目時毫無懸念地拿下滿分！


In [ ]:
# 13.4.4 學生實作練習：模擬多筆字串長度統計器
# 任務說明：實作 process_mock_stream(stream_text) 函式
# 傳入包含多行文字的 stream_text，請利用 io.StringIO 模擬標準輸入
# 使用 while True 搭配 try-except EOFError 逐行讀取
# 統計所有讀入行的「字元總數（不計換行符號）」，並回傳總字元數！

import io
import sys

def process_mock_stream(stream_text: str) -> int:
    old_stdin = sys.stdin
    sys.stdin = io.StringIO(stream_text)
    
    char_count = 0
    # 請在此處實作 while True 搭配 try-except EOFError 讀取
    while True:
        try:
            line = input()
            char_count += len(line)
        except EOFError:
            break
            
    sys.stdin = old_stdin
    return char_count

# 測試用例
mock_data = "Hello\nAPCS\nPython\n"
print("統計總字元數:", process_mock_stream(mock_data))


In [ ]:
# 13.4.4 單元測試驗證
assert process_mock_stream("abc\ndef\n") == 6
assert process_mock_stream("12345\n") == 5
assert process_mock_stream("") == 0
assert process_mock_stream("a\nb\nc\nd\n") == 4
print("13.4.4 單元測試全數通過！")


### 13.4.5 輔助區塊 else 與 finally 語意

完整的 Python 例外處理語法其實由四大家族組成：**`try`、`except`、`else`、`finally`**。理解 `else` 與 `finally` 的精準職責，能讓我們的程式架構更加嚴謹：

1. **`else` 區塊（沒有出錯才執行）**：
   - 當且僅當 `try` 區塊內**「完全沒有拋出任何例外」**時，直譯器才會執行 `else` 區塊。
   - 它的核心價值在於「區隔危險程式碼與普通業務邏輯」。我們只在 `try` 裡面放那句真正會引爆錯誤的關鍵行（例如型態轉換），而把後續一連串複雜的正常運算放在 `else` 區塊中，避免後續運算意外拋出同名例外時被錯誤攔截。
2. **`finally` 區塊（無論如何必定執行）**：
   - 無論 `try` 裡面是正常執行完畢、被 `except` 攔截補救、甚至拋出了完全沒被攔截的未處理例外、又或者是在 `try` 中途執行了 `return`，直譯器在離開這個結構前，**「百分之百保證一定會執行 `finally` 區塊」**！
   - `finally` 最常用於釋放貴重資源，例如關閉檔案、斷開連線或還原全域狀態設定。


In [ ]:
# 13.4.5 程式碼演示：try-except-else-finally 全家福執行全景
def full_suite_demo(data_input):
    print(f"\n--- 測試輸入: {repr(data_input)} ---")
    try:
        print("  [1. try] 正在嘗試執行危險轉型...")
        val = int(data_input)
    except ValueError:
        print("  [2. except] 捕捉到 ValueError，執行補救！")
        val = -1
    else:
        print(f"  [3. else] 太棒了！完全沒有出錯，安全執行後續操作：val 平方 = {val ** 2}")
    finally:
        print("  [4. finally] 🔔 無論成功或失敗，必定執行此處之清理作業！")
        
    return val

# 情況 A: 成功路徑 (try -> else -> finally)
full_suite_demo("8")

# 情況 B: 出錯路徑 (try -> except -> finally，注意 else 不會被執行)
full_suite_demo("error_string")


### 13.4.5 語法重點回顧與核心觀念提煉

四大家族的分工心智模型口訣：
- **`try`**：動手試試看（只包覆真正有風險的指令）。
- **`except`**：要是搞砸了就來這裡（處理特定的例外災情）。
- **`else`**：如果一切順利就來這裡（執行成功後的後續工作）。
- **`finally`**：不論結果如何，最後一定要來這裡收拾殘局（永遠執行的清理保證）。

在 APCS 考場實作題中，多數場景使用 `try-except` 即可滿足需求；但理解 `else` 能大幅提升程式碼的健壯度，避免 `try` 區塊過度臃腫。


In [ ]:
# 13.4.5 學生實作練習：狀態復原防禦器
# 任務說明：實作 calculate_with_flag(flag_container, func) 函式
# 傳入 flag_container 為一個包含單一布林值的串列 [False]
# 函式開始時將 flag_container[0] 設為 True（代表系統處於運作狀態）
# 嘗試執行傳入的 func() 函式並取得其回傳值
# 嚴格要求：無論 func() 執行成功還是中途拋出任何例外崩潰，
# 都必須利用 finally 確保在函式退出前將 flag_container[0] 確實還原為 False！

def calculate_with_flag(flag_container: list, func):
    flag_container[0] = True
    try:
        return func()
    except Exception:
        return None
    finally:
        # 請確保此處無論如何都會將 flag 還原為 False
        flag_container[0] = False

# 測試用例
status = [False]
success_res = calculate_with_flag(status, lambda: 10 + 20)
print("成功執行回傳:", success_res, "最終狀態是否還原:", status[0] == False)

fail_res = calculate_with_flag(status, lambda: 1 / 0)
print("出錯執行回傳:", fail_res, "最終狀態是否還原:", status[0] == False)


In [ ]:
# 13.4.5 單元測試驗證
container = [False]
res1 = calculate_with_flag(container, lambda: "APCS")
assert res1 == "APCS"
assert container[0] == False

res2 = calculate_with_flag(container, lambda: int("bad"))
assert res2 is None
assert container[0] == False
print("13.4.5 單元測試全數通過！")


### 13.4.6 競賽中的例外使用規範：何時該用 if 防守？何時該用 try-except？

學完了 `if` 條件式防守與 `try ... except` 例外捕捉之後，許多學生在解題時常產生選擇困難：「既然 `try-except` 這麼好用，我是不是乾脆所有的邊界判斷、索引越界全部都用 `try-except` 包起來就好了？」

答案是否定的！在競技程式設計中，有一條明確的**「效能與可讀性分工準則」**：

1. **優先使用 `if` 的場景（常規流程與可預測邊界）**：
   - **陣列越界檢查**：`if 0 <= i < len(arr):`
   - **除以零防範**：`if divisor != 0:`
   - **字典鍵是否存在**：`if key in d:` 或直接用 `d.get(key)`
   - *原因*：在 Python 內部，拋出並生成一個完整的例外物件（Exception Object）伴隨著收集呼叫堆疊（Traceback）的額外負擔。如果一個迴圈要執行 100 萬次，每次都靠引爆例外來跳轉，速度會比單純的 `if` 慢上 10 到 50 倍，極易引爆 TLE！
2. **必須使用 `try-except` 的場景（外部突發與不可預測事件）**：
   - **未知行數測資讀取**：`except EOFError:`（無法事先用 if 判斷輸入串流何時斷流）。
   - **字串複雜格式剖析**：嘗試將測資字串轉為整數或浮點數時，先驗字串檢查過於繁瑣且耗時。
   - **深層遞迴防爆**：捕捉 `RecursionError` 等全域直譯器限制。

簡言之：「**邊界與邏輯用 `if`，輸入與突發用 `try`**」！


In [ ]:
# 13.4.6 程式碼演示：頻繁引發例外的效能代價 vs 條件檢查
import time

REPEAT_TIMES = 100000

# 測試 A: 使用 try-except 依賴崩潰跳轉（效能巨坑）
start_a = time.time()
count_a = 0
for i in range(REPEAT_TIMES):
    try:
        # 模擬頻繁出錯引爆例外
        if i % 2 == 0:
            raise ValueError
        count_a += 1
    except ValueError:
        pass
time_a = time.time() - start_a

# 測試 B: 使用 if 條件式進行邏輯分支（極速流暢）
start_b = time.time()
count_b = 0
for i in range(REPEAT_TIMES):
    if i % 2 != 0:
        count_b += 1
time_b = time.time() - start_b

print(f"依賴例外跳轉耗時: {time_a:.4f} 秒")
print(f"使用條件判斷耗時: {time_b:.4f} 秒")
print(f"結論：在迴圈高頻執行時，if 判斷比頻繁拋出捕捉例外快了數倍以上！")


### 13.4.6 語法重點回顧與核心觀念提煉

競賽防禦雙軌策略總結：
1. **常態性邊界防禦堅決使用 `if`**：索引存取、數值比較、長度確認，通通交給 `if` 衛語句處理，既乾淨又高效。
2. **例外機制留給真突發事件**：`try-except` 是我們的「安全保險絲」，專門防範不可抗力的外部輸入中斷（如 `EOFError`）或罕見的邊界格式異常。
3. **兩者相輔相成**：前線有 `if` 進行高速篩選，後方有 `try-except` 保底兜底，形成滴水不漏的雙重防線！


In [ ]:
# 13.4.6 學生實作練習：雙軌防禦數值處理器
# 任務說明：實作 robust_batch_processor(token_list) 函式
# 傳入 token_list 為一個字串串列
# 需求：
# 1. 逐一處理每個 token
# 2. 如果 token 是空字串 ""，請使用 if 條件式提前略過（不要依賴例外）
# 3. 否則嘗試使用 int(token) 轉為整數累加至 total
# 4. 若 int() 轉型失敗（引發 ValueError），精準捕捉並略過
# 回傳計算出的合法整數總和！

def robust_batch_processor(token_list: list) -> int:
    total = 0
    # 請結合 if 與 try-except 進行雙軌處理
    for token in token_list:
        if not token:
            continue
        try:
            total += int(token)
        except ValueError:
            pass
    return total

# 測試用例
data = ["10", "", "20", "bad", "", "30"]
print("處理總和:", robust_batch_processor(data))


In [ ]:
# 13.4.6 單元測試驗證
assert robust_batch_processor(["1", "2", "3"]) == 6
assert robust_batch_processor(["", "", ""]) == 0
assert robust_batch_processor(["10", "abc", "", "20"]) == 30
assert robust_batch_processor([]) == 0
print("13.4.6 單元測試全數通過！")


## 13.4 總結與例外防禦決策指南

在本單元中，我們完整建構了 Python 原生的例外防禦架構，並掌握了 APCS 競賽中最關鍵的未知行數讀檔神技。以下為核心重點精華摘要：

| 關鍵語法 / 概念 | 核心功能與職責 | APCS 考場最佳實踐規範 |
| :--- | :--- | :--- |
| **`try ... except`** | 嘗試執行危險代碼，捕捉崩潰例外 | 專用於處理不可預測的外部輸入或複雜轉型 |
| **`except ExceptionType:`** | 精準指定攔截特定例外類別 | **嚴禁裸露 `except:`**，務必指名以防遮蔽 NameError |
| **`while True + EOFError`** | 讀取未知行數測資至檔案結束 | **APCS 必背萬用通關模板**，遇 EOFError 立即 break |
| **`else` 區塊** | 只有在 `try` 完全無出錯時才執行 | 保持 `try` 精簡，將正常後續運算放於 else |
| **`finally` 區塊** | 無論是否出錯必定在退場前執行 | 保證全域狀態還原或資源安全釋放 |
| **`if` vs `try-except`** | LBYL（事先檢查） vs EAFP（事後補救） | 迴圈內頻繁邊界用 `if`，未測資料讀取用 `try` |

### 🚀 下一步學習指引
至此，我們已經徹底防禦了靜態語法錯誤（CE）與執行中途崩潰（RE）。然而，在競賽中往往有另一種最令人崩潰的情況：**程式順利執行完畢、沒有跳出任何紅色報錯，但上傳 OJ 後卻持續拿到刺眼的 WA（Wrong Answer，答案錯誤）！**
在下一單元 **13-5《語意錯誤（Logic Error）與常見邏輯盲點排查（WA 防範）》** 中，我們將地毯式剖析差一錯誤、運算優先級、浮點數精準度、以及淺拷貝幽靈修改等五大深層邏輯暗坑！
